In [ ]:
import sympy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

import dataset
import dataset_feynman1d
import dataset_feynman2d
import dataset_feynmannd
import dataset_physics
import dataset_misc1d
import dataset_misc2d
import dataset_misc3d
import dataset_miscnd
import space
from gp import gp
from gp import creator as gp_creator
from gp import evaluator as gp_evaluator, selector as gp_selector
from gp import crossover as gp_crossover, mutator as gp_mutator
from symbols import syntax_tree, visitor
import randstate

In [ ]:
SAMPLE_SIZE      = 150
TRAIN_SIZE       = 0.7
NOISE            = 0 #0.05

POPSIZE          = 500
MAX_STREE_DEPTH  = 8
MAX_STREE_LENGTH = 20
GENERATIONS      = 50
GROUP_SIZE       = 3  # tournament selector.
MUTATION_RATE    = 0.15
ELITISM          = 1

RANDSTATE = 1234

In [ ]:
randstate.setstate(RANDSTATE)

#S = dataset_misc1d.MagmanDataset()
#S = dataset_misc1d.Nguyen7()
#S = dataset_misc1d.R1()
#S = dataset_misc1d.R2()
S = dataset_misc2d.Resistance2()
#S = dataset_misc3d.Resistance3()
#S = dataset_misc1d.ABSDataset()
#S = dataset_misc3d.Gravity()
#S = dataset_feynman2d.FeynmanICh6Eq20()
#S = dataset_feynman1d.FeynmanIICh27Eq16()
#S = dataset_feynman1d.FeynmanIICh27Eq16()
#S = dataset_physics.RocketFuelFlow()
#S = dataset_physics.AircraftLift()
#S = dataset_misc2d.Pagie1()
#S = dataset_miscnd.WavePower()
#S = dataset_feynmannd.FeynmanIICh6Eq15a()

S.sample(size=SAMPLE_SIZE, noise=NOISE, mesh=False)
#S.load('../data/magman.csv')

S.split(train_size=TRAIN_SIZE)
#S.get_plotter().plot(width=8, height=6, plot_knowldege=False)

S_train = dataset.NumpyDataset(S)
S_test  = dataset.NumpyDataset(S, test=True)

In [ ]:
np.seterr(all='ignore')

syntax_tree.SyntaxTreeInfo.set_problem(S_train)

solutionCreator = gp_creator.PTC2RandomSolutionCreator(nvars=S.nvars, const_prob=0)

multiMutator = gp_mutator.MultiMutator(
      gp_mutator.SubtreeReplacerMutator(MAX_STREE_DEPTH, MAX_STREE_LENGTH, solutionCreator),
      gp_mutator.FunctionSymbolMutator(),
      gp_mutator.NumericParameterMutator(all=True),
      gp_mutator.NumericParameterMutator(all=False)
      )

evaluator = gp_evaluator.BackscaleMSEEvaluator(S_train)
selector  = gp_selector.TournamentSelector(GROUP_SIZE)
crossover = gp_crossover.BackscaleSubTreeCrossover(MAX_STREE_DEPTH, MAX_STREE_LENGTH, evaluator, min_sparsity=0.7)

settings = gp.GPSettings(
      POPSIZE, GENERATIONS, MAX_STREE_DEPTH, MAX_STREE_LENGTH, S_train, S_test,
      creator=solutionCreator,
      evaluator=evaluator,
      selector=selector,
      crossover=crossover,
      mutator=multiMutator,
      mutrate=MUTATION_RATE,
      elitism=ELITISM,
      knowledge=S.knowledge,
      corrector=None)
symb_regressor = gp.BackscaleGP(settings)

with tqdm(total=symb_regressor.ngen-1) as pbar:
      def on_newgen(genidx, status):
            pbar.update(1)
            pbar.set_description(status)
      best_stree, best_eval = symb_regressor.evolve(newgen_callback=on_newgen)

for i in range(10):
      print(symb_regressor.population[i])
best_stree = symb_regressor.population[1]

In [ ]:
test_data_evaluator = gp_evaluator.NMSEEvaluator(S_test)
best_stree.clear_output()
print("\n--- Best syntax tree ---")
print(best_stree)
print(best_eval)
print(f"Max depth: {best_stree.get_max_depth()}")
print(f"Length: {best_stree.get_nnodes()}")
print(f"Test NMSE: {test_data_evaluator.evaluate(best_stree).value}")

sparsity_calculator = visitor.BackscaleSparsityCalculator()
best_stree.accept(sparsity_calculator)
print(f"Sparsity: {sparsity_calculator.get_sparsity()}")

best_stree.clear_output()
#S.get_plotter().plot(width=8, height=6, plot_knowldege=False, model=best_stree, zoomout=1)

print()
backscale_printer = visitor.BackscalePrinter()
best_stree.accept(backscale_printer)

In [ ]:
symb_regressor.stats.plot()